In [ ]:
%pip install -q dotenv llama_stack_client==0.4.2

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from dotenv import load_dotenv

from llama_stack_client import LlamaStackClient

In [ ]:
def stream_response(stream):
    """Stream responses API events with MCP tool call support."""
    for event in stream:
        event_type = getattr(event, "type", None)

        if event_type == "response.output_text.delta":
            print(event.delta, end="", flush=True)

        elif event_type == "response.refusal.delta":
            print(event.delta, end="", flush=True)

        # MCP tool call events
        elif event_type == "response.output_item.added":
            item = getattr(event, "item", None)
            if item and getattr(item, "type", None) == "mcp_call":
                print(f"\n🔧 MCP call: {getattr(item, 'name', '')}")
        elif event_type == "response.mcp_call.in_progress":
            print("  executing...")
        elif event_type == "response.mcp_call.completed":
            print("  completed")
        elif event_type == "response.mcp_call.failed":
            print("  failed")

    print()

In [ ]:
load_dotenv()
base_url = os.getenv("REMOTE_BASE_URL", "http://localhost:8321")

client = LlamaStackClient(base_url=base_url)

# Get MCP server URLs from registered toolgroups
toolgroups = client.toolgroups.list()
mcp_tools = []
for tg in toolgroups:
    if tg.mcp_endpoint:
        label = tg.identifier.replace("mcp::", "")
        print(f"  Found MCP toolgroup: {tg.identifier} -> {tg.mcp_endpoint.uri}")
        mcp_tools.append({
            "type": "mcp",
            "server_label": label,
            "server_url": tg.mcp_endpoint.uri,
        })

In [ ]:
MODEL = "vllm/qwen3-8b"
INSTRUCTIONS = """You are a helpful customer service assistant. 
Use the available MCP tools to search customers, fetch orders and invoices.
Always return the response in a friendly and helpful tone."""

TOOLS = mcp_tools

In [ ]:
question = "find more detail about franwilson@example.com"

stream = client.responses.create(
    model=MODEL,
    input=question,
    instructions=INSTRUCTIONS,
    tools=TOOLS,
    stream=True,
)

stream_response(stream)

In [ ]:
question = "find all order and invoice about franwilson@example.com"

stream = client.responses.create(
    model=MODEL,
    input=question,
    instructions=INSTRUCTIONS,
    tools=TOOLS,
    stream=True,
)

stream_response(stream)